# 05 · Validation Plan — SPR/BLI + isoform-specificity panel + nucleotide-state test + controls

**Standard slot:** *validation plan.* **For Project 08 this means:** turn the top candidates into a
**costed, controlled wet-lab plan** — SPR/BLI affinity vs KRAS, an **isoform-specificity panel**
(KRAS/HRAS/NRAS), a **nucleotide-state-dependence test** (GDP- vs GppNHp-loaded KRAS) `[stretch]`, the
mandatory controls (positive known KRAS binder, **scrambled-interface** negative, unrelated negative),
an expression strategy (nucleotide-loaded reagent), and the **Boltz-2 affinity** stretch (scaffold only)
(D4/D5).

A design that passes every filter is a **hypothesis** — SPR/BLI + the isoform panel are what test it.
Needs `results/top_candidates.csv` (notebook 04).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Draft the experimental validation plan

Generate a plan card from the top candidates: assays, the isoform panel, the nucleotide-state test,
controls, expression, timeline, costed reagents. Fill the `<...>` from your own numbers; this is the
deliverable other people will actually read.

In [ ]:
import pandas as pd, os

top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
n_top = len(top)
by_par = top.groupby("paradigm").size().to_dict() if n_top else {}

plan = f"""# KRAS Binder Validation Plan (Project 08 — by <your name>, <date>)

## Candidates
Top {n_top} candidates carried forward ({by_par}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured — `pae_interaction` is confidence (not affinity),
and the in-silico selectivity gap is NOT measured selectivity. No K_D is reported here.

## Expression strategy
- Binders: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (50-90 aa) -> high yield expected.
- KRAS reagent: express the G-domain (res ~1-169); NUCLEOTIDE-LOAD it deliberately -> prepare BOTH
  GDP-loaded and GppNHp/GMPPCP-loaded KRAS (keep Mg2+). Confirm each is folded/active before testing binders.
- For the isoform panel: express/obtain HRAS and NRAS G-domains under the SAME conditions.

## Assays (go/no-go -> basic -> selectivity/functional)
1. Go/no-go: express -> SDS-PAGE -> SEC (monodisperse?).
2. Affinity: SPR or BLI vs immobilized KRAS -> K_D + kinetics (k_on/k_off). Test a dilution series.
3. ISOFORM-SPECIFICITY PANEL (the centerpiece): run the SAME binder vs KRAS, HRAS, and NRAS under
   identical conditions -> quantify selectivity. A pan-RAS binder is a weaker result; report it.
4. NUCLEOTIDE-STATE TEST [stretch]: compare binding to GDP-loaded vs GppNHp-loaded KRAS -> a
   state-specific binder should discriminate.
5. Functional (extension): effector competition -> does the binder block RAF-RBD binding to KRAS-GTP?
6. Stability: DSF (Tm). Deep (optional): co-crystal / cryo-EM; cell-based KRAS-pathway readout.

## Controls (MANDATORY)
- Positive: a known KRAS binder (published DARPin/monobody/binder, or a G12C-inhibitor complex as a
  state reference) -> assay + KRAS reagent are active.
- Negative (scrambled-interface): YOUR OWN top design with its interface residues scrambled/mutated
  -> must LOSE binding (cleanest specificity control).
- Negative (unrelated): an unrelated mini-protein of similar size -> should not bind.
- Isoform off-targets (HRAS/NRAS) double as the selectivity readout AND a specificity control.

## Realistic expectations
KRAS is a hard target; SELECTIVITY (isoform + allele) is harder still. In-silico hit rates vary widely
and the MAJORITY of in-silico hits fail experimentally. Expect to test many to find a few real, and
fewer still selective, binders. Report the experimental hit rate AND the measured selectivity honestly.
Do NOT imply a working/selective binder or fabricate a K_D.

## Timeline + costed reagents (fill in)
- Gene synthesis ({n_top} binders + scrambled-interface negatives): $<...>, <...> weeks (IGSC-screened provider).
- KRAS/HRAS/NRAS reagents + nucleotide loading (GDP, GppNHp) + SPR/BLI chips + positive control: $<...>.
- Personnel/instrument time: <...> weeks.

## Responsible research
Inhibitory/blocking binders to KRAS, a human oncotarget, for cancer therapeutics/diagnostics (in scope).
Gene synthesis via a biosecurity-screening provider; wet lab under institutional biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill the <...> placeholders from your numbers.")
print(plan[:600], "...")

## 2 · Build the scrambled-interface negative controls

The single cleanest specificity control: take each top design and **scramble its interface residues**
(the positions contacting KRAS) — it should **lose** binding. Generating these alongside the real designs
(same expression batch) makes the SPR/BLI comparison airtight. Here we scaffold the sequence-level
scramble deterministically; on Colab, scramble the *interface* positions specifically using the predicted
contacts.

In [ ]:
import random
import binder_tools as bt   # bt._hashints gives a DETERMINISTIC seed (Python's hash() is salted)

def scramble_interface(seq, frac=0.4, seed=0):
    """Deterministically shuffle a fraction of the sequence as a NEGATIVE-CONTROL stand-in.
    On Colab, scramble the predicted INTERFACE residues specifically (positions contacting KRAS)."""
    rng = random.Random(seed)
    seq = list(seq)
    idx = list(range(len(seq)))
    rng.shuffle(idx)
    k = max(1, int(len(seq) * frac))
    chosen = idx[:k]
    vals = [seq[i] for i in chosen]
    rng.shuffle(vals)
    for i, v in zip(chosen, vals):
        seq[i] = v
    return "".join(seq)

negs = []
if n_top and "sequence" in top.columns:
    for _, r in top.iterrows():
        s = str(r.get("sequence", ""))
        if s and s != "nan":
            negs.append(dict(design_id=str(r["design_id"]) + "_SCRAM",
                             parent=r["design_id"], paradigm=r.get("paradigm"),
                             sequence=scramble_interface(s, seed=bt._hashints(r["design_id"]) % 10**6),
                             role="scrambled-interface negative control"))
    pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
    print(f"wrote results/negative_controls.csv: {len(negs)} scrambled-interface negatives")
else:
    print("Run notebook 04 first to produce results/top_candidates.csv with sequences.")

## 3 · (Stretch) Boltz-2 affinity on top hits `[stretch]`

Boltz-2 can predict a binding-affinity signal for the top complexes. Use it for **relative ranking +
caveats only** — **never fabricate a K_D**, and never present a predicted number as measured. This tells
you which hits to test first, not whether they bind (or are selective).

In [ ]:
# Scaffold ONLY. Do NOT invent affinities. On Colab:
#   pip install boltz; build the (binder, KRAS) complex input; run boltz predict with affinity mode;
#   read the predicted-affinity signal and report the RELATIVE ranking of the top hits + heavy caveats.
#   You can also run it per isoform to prioritize the most KRAS-selective hits (still relative-only).
# Pinned upstream (verify): https://github.com/jwohlwend/boltz
print("Boltz-2 affinity is a STRETCH scaffold: relative ranking + caveats only, NEVER a fabricated K_D.")
print("Use it to PRIORITIZE which top hits to test first in SPR/BLI + the isoform panel — not as evidence of binding.")

## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: SPR/BLI + **isoform-specificity panel** + **nucleotide-state test**, expression (nucleotide-loaded KRAS), timeline, costed reagents.
- [ ] Controls specified: positive (known KRAS binder), **scrambled-interface** negative (`results/negative_controls.csv`), unrelated negative; HRAS/NRAS off-targets double as the selectivity readout.
- [ ] (Stretch) Boltz-2 affinity used only for relative ranking, with caveats — no fabricated K_D.
- [ ] Honest framing: every design is a hypothesis until SPR/BLI + the isoform panel; report the experimental hit rate AND measured selectivity.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a rigorous, honestly-reported KRAS binder campaign whose centerpiece is **selectivity**.